In [ ]:
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
import xarray as xr

In [ ]:
from pcr import helper, storm, slr, erosion, shoreline

In [ ]:
# simple look-up function to get lambda for a given month
def lam(day_t, lams, date_start):
    '''
    Get the intensity (lambda) for a given time in days.
    day_t       : float
    lams        : list or array of float of intensity values for each month (12 values).
    date_start  : np.datetime64 of the start date corresponding to day_t = 0.
    Returns the intensity (lambda) for the month corresponding to day_t.
    '''
    date_t = date_start + np.timedelta64(int(day_t*24), 'h')
    
    month = date_t.item().month

    return lams[month-1]

def simulate_nhpp_thinning(T, lams, date_start):
    """
    Simulate a Non-Homogeneous Poisson Process on [0, T]
    using the thinning method.

    Inputs:
        T           : final time
        lams        : array of λ(t) giving the rate at time t
        date_start  : day the simulation starts (numpy datetime64)

    Output:
        arrivals    : numpy array of arrival times in [0, T]
    """

    lambda_max = lams.max()  
    t = 0.0
    arrivals = []

    while True:
        # Step 1: propose next arrival in Poisson(λ_max)
        # Gap ~ Exponential(λ_max)
        gap = np.random.exponential(1.0 / lambda_max) * 365.25  # convert to days
        t = t + gap
        if t > T:
            break

        # Step 2: accept with probability λ(t) / λ_max
        u = np.random.uniform(0.0, 1.0)
        if u <= lam(t, lams, date_start) / lambda_max:
            arrivals.append(t)

    return np.array(arrivals)

def go_simulate_multiple(ds, n_sims, method):
    # detect storm 
    ts_hs = 95 
    ts_dur = 12 

    hs, dir, tp, time = helper.era5_input(ds)

    detected_storm, _ = storm.detect(hs, dir, tp, time, ts_hs, ts_dur, ts_between=48)

    # fit storm and gap 
    fitted_storms = storm.fit_storm(detected_storm)
    lambda_storm_gap = storm.fit_lambda_gap(detected_storm, fillna='zeros')
    lambda_storm = storm.fit_lambda(detected_storm, fillna='zeros')

    # generate hs, dur, tp, direction sample
    n_sampling = detected_storm.shape[0] * n_sims

    _, durs, _, _ = storm.generate(
        fitted_storm=fitted_storms, 
        sampling_size=n_sampling, 
        oversample=0.1, 
        max_dur=np.max(detected_storm.duration))

    # Define the time horizon (same as the data period)
    # date_start = np.datetime64('1979-01-01')
    # date_end = np.datetime64('2020-01-01')
    date_start = ds.valid_time[0].values.astype('datetime64[s]')
    date_end = ds.valid_time[-1].values.astype('datetime64[s]')

    t_days = (date_end-date_start).item().days # time horizon in days
    
    # durs = storms_sample['duration'].values
    res = []
    durs_sim = []
    # sample_idx = []
    counter = 0

    for i in range(n_sims):
        dur = durs[counter:]
        if method=='nhpp':
            res.append(simulate_nhpp_thinning(t_days, lambda_storm, date_start))
        elif method == 'modified': 
            res.append(storm.gap_nhpp_thinning(t_days, lambda_storm_gap, date_start, dur)) # in days since date_start
        
        storm_count = len(res[i])
        durs_sim.append(dur[:storm_count])
        
        counter += storm_count
        
    return res, durs_sim, detected_storm


In [ ]:
# bar chart per month 
def compare_bar_month(dt_1:np.array, dt_2:np.array, label_1: str='Data 1', label_2: str='Data 2', title:str=''):
    
    
    df_1 = pd.DataFrame({'start': dt_1})
    df_2 = pd.DataFrame({'start': dt_2})

    count_sim = pd.DataFrame(
        index=np.arange(1,13),
    )

    count_sim[label_1] = df_1.groupby(by=df_1['start'].dt.month).count()
    count_sim[label_2] = df_2.groupby(by=df_2['start'].dt.month).count()

    # Plot
    fig, ax = plt.subplots(figsize=(10,5))
    width = 0.4
    x = np.arange(1, 13)

    ax.grid(linestyle='--', alpha=0.3, zorder=0)
    ax.bar(x - width/2, count_sim[label_1], width, label="Simulation")
    ax.bar(x + width/2, count_sim[label_2], width, label="Data")

    ax.set_xticks(x)
    ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                         'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    ax.set_xlabel("Month")
    ax.set_ylabel("Number of Storms")
    ax.set_title(title)
    ax.legend()

    plt.show()

    return fig, ax

# compare empirical CDF 
def get_gap(storm_start, durs):
    
    end_storm = storm_start + (durs / 24)
    gap = storm_start[1:] - end_storm[:-1]
    
    return gap 

def compare_ecdf(storm_start, durs, detected_storm_):

    # CDF of gaps
    if isinstance(storm_start, list):
        gap_sim = np.concatenate([get_gap(res, dur_pairs) for res, dur_pairs in zip(storm_start, durs)])
    else: 
        gap_sim = get_gap(storm_start, durs)

    gap_data = detected_storm_['gap'][1:].values

    # Sort for CDF
    gap_sim = np.sort(gap_sim)
    gap_data = np.sort(gap_data)

    # Empirical CDFs
    cdf_sim = np.arange(len(gap_sim)) / len(gap_sim)
    cdf_data = np.arange(len(gap_data)) / len(gap_data)

    # Plot
    fig, ax = plt.subplots(figsize=(10,6))

    ax.plot(gap_data, cdf_data, label='Data')
    ax.plot(gap_sim, cdf_sim, '--', label='Simulated')

    ax.grid(linestyle='--', alpha=0.3, zorder=0)
    # ax.set_title("")
    ax.set_xlabel('Gaps (days)')
    ax.set_ylabel('CDF')
    ax.legend()

    plt.show()

    return fig, ax, gap_sim

# compare storm per simulation 
def plot_storm_count(res, detected_storm):
    N = [result.size for result in res]

    # plot 
    fig, ax = plt.subplots(figsize=(8,5))
    
    ax.hist(N, bins=50, alpha=0.7, color='tab:blue', edgecolor='black', label='NHPP Thinning Simulation')
    ax.axvline(detected_storm.shape[0], color='red', linestyle='dashed', linewidth=2, label='Observed Data')
    ax.set_xlabel("Number of Storms in 41 years")
    ax.set_ylabel("Frequency")
    ax.set_title("Histogram of Number of Storms in 10,000 NHPP Thinning Simulations")
    ax.text(0.02, 0.87, f'median: {np.median(N)}\nmean: {np.mean(N)}\ndata: {detected_storm.shape[0]}', 
            transform=ax.transAxes, 
            )
    ax.legend()

    plt.show()

    return fig, ax

# compare monthly bar chart of storm count 
def avg_bar_chart(res, date_start, detected_storm):
    
    numsim = len(res)

    # get the average number of storms per month from the simulations
    count_sim = pd.DataFrame(
        {'count': np.zeros(12)},
        index=np.arange(1,13),
    )

    for i in range(numsim):
        out_sl = pd.DataFrame({
            'time':[date_start + np.timedelta64(int(start*24), 'h') for start in res[i]], 
        })

        out_sl['month'] = [s_dt.month for s_dt in out_sl['time']]

        count_sim['addition'] = out_sl.groupby('month').count()['time']
        count_sim['addition'] = count_sim['addition'].fillna(0)
        count_sim['count'] = count_sim['count'] + count_sim['addition']

    count_sim['avg_count'] = count_sim['count'] / numsim

    # get the count for detected storm (data)
    storm_start_data = pd.DataFrame(
        {
            'date_start': helper.datenum_to_datetime(detected_storm['start'])
        }
    )

    count_sim['data_count'] = storm_start_data.groupby(by=storm_start_data['date_start'].dt.month).count()

    # plot the monthly counts 
    fig, ax = plt.subplots(figsize=(10, 5))
    width = 0.4
    x = np.arange(1, 13)

    ax.grid(linestyle='--', alpha=0.3, zorder=0)
    ax.bar(x-width/2, count_sim['avg_count'], width, label= "10,000 Simulation")
    ax.bar(x+width/2, count_sim['data_count'], width, label= "Data")
    ax.set_xticks(x)
    ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    ax.set_xlabel("Month")
    ax.set_ylabel("Number of Storms")
    ax.set_title("Monthly Storm Counts: 10,000 HNPP Thinning Simulation vs Data")
    ax.legend()

    plt.show()

    return fig, ax


# NHPP Method

## Station P2

In [ ]:
# access from the pre-downloaded 
station = 'p2'
data_path = f'../data/ERA5/ts/unzipped/{station}_ts.nc'
ds_p2 = xr.open_dataset(data_path)

In [ ]:
res_p2, durs_sim_p2, detected_storm_p2  = go_simulate_multiple(ds_p2, 10000, 'nhpp')

In [ ]:
fig, ax = plot_storm_count(res_p2, detected_storm_p2)

In [ ]:
date_start = ds_p2.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_p2, date_start, detected_storm_p2)

In [ ]:
fig, ax = compare_ecdf(res_p2, durs_sim_p2, detected_storm_p2)

In [ ]:
gap_sim_p2 = np.concatenate([get_gap(res, dur_pairs) for res, dur_pairs in zip(res_p2, durs_sim_p2)])
negative_gap = gap_sim_p2[gap_sim_p2<0]

print(f'non-positive gaps: {gap_sim_p2[gap_sim_p2<0]}, ({len(negative_gap) / len(gap_sim_p2):.2f})')

## Station P11

In [ ]:
# access from the pre-downloaded 
station = 'p11'
data_path = f'../data/ERA5/ts/unzipped/{station}_ts.nc'
ds_p11 = xr.open_dataset(data_path)

In [ ]:
res_p11, durs_sim_p11, detected_storm_p11  = go_simulate_multiple(ds_p11, 10000, 'nhpp')

In [ ]:
fig, ax = plot_storm_count(res_p11, detected_storm_p11)

In [ ]:
date_start = ds_p11.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_p11, date_start, detected_storm_p11)

In [ ]:
fig, ax = compare_ecdf(res_p11, durs_sim_p11, detected_storm_p11)

In [ ]:
gap_sim_p11 = np.concatenate([get_gap(res, dur_pairs) for res, dur_pairs in zip(res_p11, durs_sim_p11)])
negative_gap = gap_sim_p11[gap_sim_p11<0]

print(f'non-positive gaps: {gap_sim_p11[gap_sim_p11<0]}, ({len(negative_gap) / len(gap_sim_p11):.2f})')

## Station P23

In [ ]:
# access from the pre-downloaded 
station = 'p23'
data_path = f'../data/ERA5/ts/unzipped/{station}_ts.nc'
ds_p23 = xr.open_dataset(data_path)

In [ ]:
res_p23, durs_sim_p23, detected_storm_p23  = go_simulate_multiple(ds_p23, 10000, 'nhpp')

In [ ]:
fig, ax = plot_storm_count(res_p23, detected_storm_p23)

In [ ]:
date_start = ds_p23.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_p23, date_start, detected_storm_p23)

In [ ]:
fig, ax = compare_ecdf(res_p23, durs_sim_p23, detected_storm_p23)

In [ ]:
gap_sim_p23 = np.concatenate([get_gap(res, dur_pairs) for res, dur_pairs in zip(res_p23, durs_sim_p23)])
negative_gap = gap_sim_p23[gap_sim_p23<0]

print(f'non-positive gaps: {gap_sim_p23[gap_sim_p23<0]}, ({len(negative_gap) / len(gap_sim_p23):.2f})')

# Modified
## P2

In [ ]:
res_p2_mod, durs_sim_p2_mod, detected_storm_p2_mod  = go_simulate_multiple(ds_p2, 10000, 'modified')

In [ ]:
fig, ax = plot_storm_count(res_p2_mod, detected_storm_p2_mod_mod)

In [ ]:
date_start = ds_p2.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_p2_mod, date_start, detected_storm_p2_mod)

In [ ]:
fig, ax, gap_sim_p2_mod = compare_ecdf(res_p2, durs_sim_p2, detected_storm_p2)

In [ ]:
gap_sim_p2_mod = np.concatenate([get_gap(res, dur_pairs) for res, dur_pairs in zip(res_p2_mod, durs_sim_p2_mod)])
negative_gap_p2 = gap_sim_p2_mod[gap_sim_p2_mod<0]

print(f'non-positive gaps: {negative_gap_p2}, ({len(negative_gap_p2) / len(gap_sim_p2_mod):.2f})')

## P11

In [ ]:
res_p11, durs_sim_p11, detected_storm_p11  = go_simulate_multiple(ds_p11, 10000, 'modified')

In [ ]:
fig, ax = plot_storm_count(res_p11, detected_storm_p11)

In [ ]:
date_start = ds_p11.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_p11, date_start, detected_storm_p11)

In [ ]:
fig, ax = compare_ecdf(res_p11, durs_sim_p11, detected_storm_p11)

In [ ]:
gap_sim_p11 = np.concatenate([get_gap(res, dur_pairs) for res, dur_pairs in zip(res_p11, durs_sim_p11)])
negative_gap = gap_sim_p11[gap_sim_p11<0]

print(f'non-positive gaps: {gap_sim_p11[gap_sim_p11<0]}, ({len(negative_gap) / len(gap_sim_p11):.2f})')

## Station P23

In [ ]:
res_p23, durs_sim_p23, detected_storm_p23  = go_simulate_multiple(ds_p23, 10000, 'modified')

In [ ]:
fig, ax = plot_storm_count(res_p23, detected_storm_p23)

In [ ]:
date_start = ds_p23.valid_time[0].values.astype('datetime64[s]')

fig, ax = avg_bar_chart(res_p23, date_start, detected_storm_p23)

In [ ]:
fig, ax = compare_ecdf(res_p23, durs_sim_p23, detected_storm_p23)

In [ ]:
gap_sim_p23 = np.concatenate([get_gap(res, dur_pairs) for res, dur_pairs in zip(res_p23, durs_sim_p23)])
negative_gap = gap_sim_p23[gap_sim_p23<0]

print(f'non-positive gaps: {gap_sim_p23[gap_sim_p23<0]}, ({len(negative_gap) / len(gap_sim_p23):.2f})')